Proyecto final Unidad 6 Filtros

Universidad de Antioquia

William Andres Mora Herrera

Jean Carlos Suarez Hernandez

In [345]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.signal as signal
import plotly.graph_objects as go

FS = 200.0
Ruta = Path("Archivos caídas\F06_SA01_R04.txt")

COLUMNAS = [
    "ADXL_X", "ADXL_Y", "ADXL_Z",
    "ITG_X",  "ITG_Y",  "ITG_Z",
    "MMA_X",  "MMA_Y",  "MMA_Z"
]

FACTORES = {
    "ADXL": (2 * 16) / (2**13),
    "ITG":  (2 * 2000) / (2**16),
    "MMA":  (2 * 8) / (2**14)
}

In [346]:
def cargar_senal(ruta=Ruta, magnitud=True):
    df = pd.read_csv(ruta, sep=',', engine='python', header=None, names=COLUMNAS)
    df['MMA_Z'] = df['MMA_Z'].astype(str).str.replace(';', '').str.strip().astype(int)
    df = df.astype(int)
    
    for prefijo, factor in FACTORES.items():
        cols = [f'{prefijo}_X', f'{prefijo}_Y', f'{prefijo}_Z']
        df[cols] = df[cols] * factor
        
    if magnitud:
        df = pd.DataFrame({
            prefijo: np.sqrt(df[f'{prefijo}_X']**2 + df[f'{prefijo}_Y']**2 + df[f'{prefijo}_Z']**2)
            for prefijo in FACTORES
        })
        
    return df

df_senal = cargar_senal()
print(f"Archivo cargado: {Ruta.name} | Filas: {len(df_senal)}")
print(df_senal.head())

Archivo cargado: F06_SA01_R04.txt | Filas: 3000
       ADXL        ITG       MMA
0  0.942734  54.145649  0.822916
1  0.954557  58.417309  0.829298
2  0.988798  60.687302  0.856301
3  1.019681  61.672092  0.888280
4  1.064867  61.690029  0.921408


In [347]:
FRECHAZO = 6.0
FPASO = 11.0
GPASS = 1.0
GSTOP = 25.0

nyq = FS / 2
orden_hpf, wn_hpf = signal.buttord(FPASO / nyq, FRECHAZO / nyq, GPASS, GSTOP)
sos_hpf = signal.butter(orden_hpf, wn_hpf, btype='highpass', output='sos')

def filtrar_senal(df, columna='ADXL'):
    return pd.Series(signal.sosfiltfilt(sos_hpf, df[columna].values), index=df.index)

senal_filtrada = filtrar_senal(df_senal, columna='ADXL')
print(f"Orden: {orden_hpf} | Pico filtrado: {senal_filtrada.abs().max():.3f} g")

Orden: 6 | Pico filtrado: 0.287 g


In [348]:
from plotly.subplots import make_subplots

def graficar_bode():
    w, h = signal.sosfreqz(sos_hpf, worN=2048, fs=FS)
    mag_db = 20 * np.log10(np.maximum(np.abs(h), 1e-12))
    mag_eff_db = 2 * mag_db
    fase_deg = np.unwrap(np.angle(h)) * (180 / np.pi)
    fc_hz = wn_hpf * (FS / 2)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
        subplot_titles=(
            f"Diagrama de Bode: Butter highpass, orden {orden_hpf}, Wn = {fc_hz:.1f} Hz",
            "Fase"
        )
    )

    fig.add_trace(go.Scatter(x=w, y=mag_db, name="Un paso (freqz)", line=dict(color="#1f77b4")), row=1, col=1)
    fig.add_trace(go.Scatter(x=w, y=mag_eff_db, name="Efectiva (ida y vuelta, filtfilt)", line=dict(color="red", dash="dash")), row=1, col=1)
    fig.add_vline(x=fc_hz, line_dash="dash", line_color="gray", annotation_text=f"Wn={fc_hz:.1f} Hz", row=1, col=1)
    fig.add_vline(x=FRECHAZO, line_dash="dot", line_color="gray", annotation_text=f"fs={FRECHAZO} Hz", row=1, col=1)
    fig.add_vline(x=FPASO, line_dash="dot", line_color="gray", annotation_text=f"fp={FPASO} Hz", row=1, col=1)

    fig.add_trace(go.Scatter(x=w, y=fase_deg, name="Fase un paso", line=dict(color="#1f77b4"), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=w, y=np.zeros_like(w), name="Fase efectiva (0°)", line=dict(color="red", dash="dash"), showlegend=False), row=2, col=1)

    fig.update_xaxes(type="log", range=[np.log10(0.1), np.log10(FS / 2)], title_text="Frecuencia (Hz)", row=2, col=1)
    fig.update_yaxes(range=[-100, 5], title_text="Magnitud (dB)", row=1, col=1)
    fig.update_yaxes(title_text="Fase (°)", row=2, col=1)
    fig.update_layout(height=550, legend=dict(orientation="h", y=1.12))
    fig.show()

def graficar_senal(df=df_senal, filtrada=senal_filtrada, ruta=Ruta, columna='ADXL'):
    clase = "ADL" if ruta.name.startswith("D") else "Caída"
    t = np.arange(len(df)) / FS

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t, y=df[columna], name="Original", line=dict(color="gray", width=1)))
    fig.add_trace(go.Scatter(x=t, y=filtrada, name="Filtrada", line=dict(color="#1f77b4", width=1.2)))
    fig.update_layout(
        title=f"{ruta.stem} ({clase}) — Pico filtrado: {filtrada.abs().max():.3f} g",
        xaxis_title="Tiempo (s)",
        yaxis_title="Aceleración (g)",
        height=350,
        hovermode="x unified"
    )
    fig.show()

graficar_bode()
graficar_senal()

In [349]:
UMBRAL_ADXL = 1.75

def detectar_caida(senal_filtrada, umbral=UMBRAL_ADXL):
    pico = float(senal_filtrada.abs().max())
    es_caida = pico > umbral
    etiqueta = "Caída" if es_caida else "ADL"
    return es_caida, etiqueta, pico

clase_real = "ADL" if Ruta.name.startswith("D") else "Caída"
es_caida, etiqueta, pico = detectar_caida(senal_filtrada)

print(f"Archivo:   {Ruta.name}")
print(f"Predicho:  {etiqueta} (Pico: {pico:.4f} g | Umbral: {UMBRAL_ADXL} g)")
print(f"Real:      {clase_real}")

Archivo:   F06_SA01_R04.txt
Predicho:  ADL (Pico: 0.2867 g | Umbral: 1.75 g)
Real:      Caída


Hasta aqui era el codigo para un unico filtro, pero pense que para el objetivo del proyecto, es mucho mas peligroso que un anciano se caiga y no sea detectado.

para eso geminis me recomendo un segundo filtro pasa bajas


Esto es otra prueba añadiendo un segundo filtro para intentar evitar los falsos negativo


In [350]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.signal as signal
import plotly.graph_objects as go

FS = 200.0
Ruta = Path("Archivos caídas\F06_SA03_R05.txt")

COLUMNAS = [
    "ADXL_X", "ADXL_Y", "ADXL_Z",
    "ITG_X",  "ITG_Y",  "ITG_Z",
    "MMA_X",  "MMA_Y",  "MMA_Z"
]

FACTORES = {
    "ADXL": (2 * 16) / (2**13),
    "ITG":  (2 * 2000) / (2**16),
    "MMA":  (2 * 8) / (2**14)
}

In [351]:
def cargar_senal(ruta=Ruta):
    df = pd.read_csv(ruta, sep=',', engine='python', header=None, names=COLUMNAS)
    df['MMA_Z'] = df['MMA_Z'].astype(str).str.replace(';', '').str.strip().astype(int)
    df = df.astype(int)
    
    for prefijo, factor in FACTORES.items():
        cols = [f'{prefijo}_X', f'{prefijo}_Y', f'{prefijo}_Z']
        df[cols] = df[cols] * factor
        df[prefijo] = np.sqrt(df[f'{prefijo}_X']**2 + df[f'{prefijo}_Y']**2 + df[f'{prefijo}_Z']**2)
        
    return df

df_senal = cargar_senal()
print(f"Archivo cargado: {Ruta.name} | Filas: {len(df_senal)}")

Archivo cargado: F06_SA03_R05.txt | Filas: 3000


In [352]:
# --- Filtro 1: Pasa-Altas (Impacto brusco) ---
FPASO = 11.0
FRECHAZO = 6.0
GPASS = 1.0
GSTOP = 25.0

nyq = FS / 2
orden_hpf, wn_hpf = signal.buttord(FPASO / nyq, FRECHAZO / nyq, GPASS, GSTOP)
sos_hpf = signal.butter(orden_hpf, wn_hpf, btype='highpass', output='sos')

# --- Filtro 2: Pasa-Bajas (Gravedad y postura en baja frecuencia) ---
FC_LPF = 1.5
orden_lpf = 4
sos_lpf = signal.butter(orden_lpf, FC_LPF / nyq, btype='lowpass', output='sos')

def filtrar_senal(df, columna='ADXL', sos=sos_hpf):
    return pd.Series(signal.sosfiltfilt(sos, df[columna].values), index=df.index)

senal_filtrada = filtrar_senal(df_senal, columna='ADXL', sos=sos_hpf)
adxl_y_lpf     = filtrar_senal(df_senal, columna='ADXL_Y', sos=sos_lpf)

print(f"HPF (orden {orden_hpf}) Pico: {senal_filtrada.abs().max():.3f} g | LPF Eje Y final: {adxl_y_lpf.iloc[-int(FS):].mean():.3f} g")

HPF (orden 6) Pico: 0.645 g | LPF Eje Y final: -0.094 g


In [353]:
from plotly.subplots import make_subplots


from plotly.subplots import make_subplots

def graficar_bode():
    w, h_hpf = signal.sosfreqz(sos_hpf, worN=4096, fs=FS)
    mag_lin = np.abs(h_hpf)  # |H(f)| en escala lineal (de 0 a 1)
    
    # Niveles lineales Hp y Ha a partir de GPASS (1 dB) y GSTOP (25 dB)
    Hp = 10 ** (-GPASS / 20)   # ~0.891
    Ha = 10 ** (-GSTOP / 20)   # ~0.056
    fc_hz = wn_hpf * (FS / 2)

    # Fase de un paso (forma en Z)
    fase_hpf = np.unwrap(np.angle(h_hpf)) * (180 / np.pi)
    fase_hpf = fase_hpf - fase_hpf[-1]

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.14,
        subplot_titles=(
            f"Plantilla de Especificaciones |H(f)| — HPF Butterworth orden {orden_hpf} (fs={FRECHAZO} Hz, fp={FPASO} Hz)",
            "Fase del Filtro Pasa-Altas (forma en Z)"
        )
    )

    # --- SUBGRÁFICA 1: PLANTILLA TIPO LIBRO (|H(f)| LINEAL CON CAJAS ROJAS) ---
    # Caja roja 1 (Banda de rechazo: f <= fs, |H| > Ha)
    fig.add_shape(type="rect", x0=0, x1=FRECHAZO, y0=Ha, y1=1.1,
                  fillcolor="rgba(220, 50, 47, 0.45)", line=dict(width=0), row=1, col=1)
    # Caja roja 2 (Banda pasante: f >= fp, |H| < Hp)
    fig.add_shape(type="rect", x0=FPASO, x1=25, y0=0, y1=Hp,
                  fillcolor="rgba(220, 50, 47, 0.45)", line=dict(width=0), row=1, col=1)

    # Filtro Ideal (escalón negro en fc_hz)
    fig.add_trace(go.Scatter(
        x=[0, fc_hz, fc_hz, 25], y=[0, 0, 1, 1],
        name="Filtro Ideal", line=dict(color="black", width=2)
    ), row=1, col=1)

    # Filtro Real (curva roja punteada pasando entre las cajas)
    fig.add_trace(go.Scatter(
        x=w, y=mag_lin,
        name="Filtro Real |H(f)|", line=dict(color="#d62728", dash="dash", width=2.5)
    ), row=1, col=1)

    # Etiquetas fs, fp, Hp, Ha
    fig.add_annotation(x=FRECHAZO, y=Ha, text=f"fs={FRECHAZO} Hz (Ha={Ha:.2f})", showarrow=True, arrowhead=1, ax=-45, ay=-20, row=1, col=1)
    fig.add_annotation(x=FPASO, y=Hp, text=f"fp={FPASO} Hz (Hp={Hp:.2f})", showarrow=True, arrowhead=1, ax=45, ay=-20, row=1, col=1)

    # --- SUBGRÁFICA 2: FASE EN Z (escala log o lineal) ---
    fig.add_trace(go.Scatter(
        x=w, y=fase_hpf, name="Fase (un paso)", line=dict(color="#1f77b4", width=2.5)
    ), row=2, col=1)

    fig.update_xaxes(range=[0, 25], title_text="Frecuencia f (Hz)", row=2, col=1)
    fig.update_yaxes(range=[-0.05, 1.15], title_text="|H(f)|", row=1, col=1)
    fig.update_yaxes(range=[-30, 570], title_text="Fase (°)", row=2, col=1)
    fig.update_layout(width=820, height=560, legend=dict(orientation="h", y=1.12))
    fig.show()

def graficar_senal(df=df_senal, filtrada=senal_filtrada, postura_y=adxl_y_lpf, ruta=Ruta):
    clase = "ADL" if ruta.name.startswith("D") else "Caída"
    t = np.arange(len(df)) / FS

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                        subplot_titles=("Filtro 1 (HPF): Impacto en alta frecuencia", "Filtro 2 (LPF): Gravedad en eje vertical Y (postura)"))

    fig.add_trace(go.Scatter(x=t, y=df['ADXL'], name="ADXL Original", line=dict(color="gray", width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=filtrada, name="ADXL HPF (Impacto)", line=dict(color="#1f77b4", width=1.2)), row=1, col=1)

    fig.add_trace(go.Scatter(x=t, y=df['ADXL_Y'], name="Eje Y Original", line=dict(color="lightgray", width=1)), row=2, col=1)
    fig.add_trace(go.Scatter(x=t, y=postura_y, name="Eje Y LPF (Gravedad)", line=dict(color="#2ca02c", width=1.5)), row=2, col=1)

    fig.update_layout(title=f"{ruta.stem} ({clase}) — Pico HPF: {filtrada.abs().max():.3f} g", height=520, hovermode="x unified")
    fig.update_yaxes(title_text="Aceleración (g)", row=1, col=1)
    fig.update_yaxes(title_text="Eje Y (g)", row=2, col=1)
    fig.update_xaxes(title_text="Tiempo (s)", row=2, col=1)
    fig.show()

graficar_bode()
graficar_senal()

In [354]:
UMBRAL_ADXL = 1.75
UMBRAL_POSTURA_Y = 0.45  # Si |ADXL_Y| cae por debajo de ~0.45 g (>63° de inclinación), está horizontal

def detectar_caida(adxl_hpf, y_lpf, umbral_g=UMBRAL_ADXL, umbral_y=UMBRAL_POSTURA_Y):
    pico_g = float(adxl_hpf.abs().max())
    n_ventana = int(FS)  # Ventana de 1 segundo
    
    y_inicio = float(y_lpf.iloc[:n_ventana].abs().mean())
    y_final  = float(y_lpf.iloc[-n_ventana:].abs().mean())
    
    # Criterio 1: Impacto fuerte en alta frecuencia
    caida_impacto = pico_g > umbral_g
    # Criterio 2: Desmayo/caída suave (inició vertical > 0.7g, terminó acostado < 0.45g y hubo movimiento > 0.20g)
    caida_postura = (y_inicio > 0.70) and (y_final < umbral_y) and (pico_g > 0.20)
    
    es_caida = caida_impacto or caida_postura
    motivo = "Impacto HPF" if caida_impacto else ("Cambio de postura LPF" if caida_postura else "Ninguno")
    etiqueta = "Caída" if es_caida else "ADL"
    return es_caida, etiqueta, pico_g, y_final, motivo

clase_real = "ADL" if Ruta.name.startswith("D") else "Caída"
es_caida, etiqueta, pico_g, y_final, motivo = detectar_caida(senal_filtrada, adxl_y_lpf)

print(f"Archivo:   {Ruta.name}")
print(f"Real:      {clase_real}")
print(f"Predicho:  {etiqueta} | Motivo: {motivo} (Pico HPF: {pico_g:.4f} g | |Y| final LPF: {y_final:.3f} g)")

Archivo:   F06_SA03_R05.txt
Real:      Caída
Predicho:  Caída | Motivo: Cambio de postura LPF (Pico HPF: 0.6447 g | |Y| final LPF: 0.094 g)


In [355]:
import re
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix

# 1. Filtrar solo los .txt reales de SisFall (ej. D01_... o F06_...), ignorando Readme.txt u otros
carpeta = Ruta.parent
archivos = sorted([a for a in carpeta.glob("*.txt") if re.match(r"^[DF]\d{2}_", a.name)])
print(f"Archivos SisFall válidos encontrados: {len(archivos)}")

registros = []
for arch in archivos:
    df_tmp = cargar_senal(arch)
    hpf_tmp = filtrar_senal(df_tmp, columna='ADXL', sos=sos_hpf)
    lpf_y_tmp = filtrar_senal(df_tmp, columna='ADXL_Y', sos=sos_lpf)
    
    n_vent = int(FS)
    registros.append({
        "archivo": arch.name,
        "y_real": 0 if arch.name.startswith("D") else 1,  # 0 = ADL, 1 = Caída
        "pico_hpf": float(hpf_tmp.abs().max()),
        "y_inicio": float(lpf_y_tmp.iloc[:n_vent].abs().mean()),
        "y_final": float(lpf_y_tmp.iloc[-n_vent:].abs().mean()),
    })

df_cv = pd.DataFrame(registros)
X_idx = np.arange(len(df_cv))
y = df_cv["y_real"].values

# 2. Validación Cruzada Estratificada (5 folds)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

preds_1f = np.zeros_like(y)
preds_2f = np.zeros_like(y)

umbrales_g_grid = np.linspace(0.5, 3.0, 51)
umbrales_y_grid = np.linspace(0.30, 0.65, 15)

for train_idx, test_idx in skf.split(X_idx, y):
    train = df_cv.iloc[train_idx]
    test = df_cv.iloc[test_idx]
    
    # --- Ajuste en TRAIN: Modelo 1 Filtro (maximiza F1-score) ---
    mejor_f1_1, mejor_ug_1 = -1, 1.75
    for ug in umbrales_g_grid:
        p_tr = (train["pico_hpf"] > ug).astype(int)
        score = f1_score(train["y_real"], p_tr, zero_division=0)
        if score > mejor_f1_1:
            mejor_f1_1, mejor_ug_1 = score, ug
            
    preds_1f[test_idx] = (test["pico_hpf"] > mejor_ug_1).astype(int)
    
    # --- Ajuste en TRAIN: Modelo 2 Filtros (HPF + LPF Postura) ---
    mejor_f1_2, mejor_ug_2, mejor_uy_2 = -1, 1.75, 0.45
    for ug in umbrales_g_grid:
        for uy in umbrales_y_grid:
            cond_imp = train["pico_hpf"] > ug
            cond_pos = (train["y_inicio"] > 0.70) & (train["y_final"] < uy) & (train["pico_hpf"] > 0.20)
            p_tr = (cond_imp | cond_pos).astype(int)
            score = f1_score(train["y_real"], p_tr, zero_division=0)
            if score > mejor_f1_2:
                mejor_f1_2, mejor_ug_2, mejor_uy_2 = score, ug, uy
                
    cond_imp_te = test["pico_hpf"] > mejor_ug_2
    cond_pos_te = (test["y_inicio"] > 0.70) & (test["y_final"] < mejor_uy_2) & (test["pico_hpf"] > 0.20)
    preds_2f[test_idx] = (cond_imp_te | cond_pos_te).astype(int)

# 3. Tabla comparativa de resultados Out-of-Fold
def resumir_metricas(nombre, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "Método": nombre,
        "Exactitud (Acc)": f"{accuracy_score(y_true, y_pred)*100:.1f} %",
        "Sensibilidad (Recall Caídas)": f"{recall_score(y_true, y_pred)*100:.1f} % ({tp}/{tp+fn})",
        "Especificidad (ADL)": f"{(tn/(tn+fp))*100:.1f} % ({tn}/{tn+fp})",
        "F1-Score": f"{f1_score(y_true, y_pred):.3f}",
        "Falsos Negativos": fn,
        "Falsos Positivos": fp
    }

df_resultados = pd.DataFrame([
    resumir_metricas("1 Filtro (Solo HPF Impacto)", y, preds_1f),
    resumir_metricas("2 Filtros (HPF + LPF Postura)", y, preds_2f)
])

display(df_resultados)

rescatados = df_cv[(y == 1) & (preds_1f == 0) & (preds_2f == 1)]["archivo"].tolist()
print(f"\nCaídas suaves rescatadas por el LPF de postura ({len(rescatados)}):", rescatados)

Archivos SisFall válidos encontrados: 68


,Método,Exactitud (Acc),Sensibilidad (Recall Caídas),Especificidad (ADL),F1-Score,Falsos Negativos,Falsos Positivos
0,1 Filtro (Solo HPF Impacto),86.8 %,95.0 % (57/60),25.0 % (2/8),0.927,3,6
1,2 Filtros (HPF + LPF Postura),98.5 %,100.0 % (60/60),87.5 % (7/8),0.992,0,1



Caídas suaves rescatadas por el LPF de postura (3): ['F03_SA01_R01.txt', 'F03_SA01_R05.txt', 'F06_SA01_R04.txt']
